# 03. 메시지(Messages) — LangGraph 대화의 기본 단위

> LangGraph 상태의 핵심은 결국 **메시지 리스트**예요. System/Human/AI/Tool 4가지 타입과 메타데이터, `add_messages` 리듀서의 동작을 명확히 이해하고 갑니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 4가지 메시지 타입의 역할과 사용 시점을 설명할 수 있어요
2. `name`, `id` 메타데이터를 활용하여 다중 사용자 환경에서 메시지를 추적할 수 있어요
3. 딕셔너리 형식(`{"role": ..., "content": ...}`)으로 메시지를 구성할 수 있어요
4. `ToolMessage`의 `tool_call_id` 매칭 규칙을 이해하고 도구 호출 전체 흐름을 구현할 수 있어요
5. Provider 네이티브 형식과 표준 `content_blocks`를 활용하여 멀티모달 메시지를 구성할 수 있어요

## 사전 지식

- 이전 노트북: `02-Models.ipynb` — `init_chat_model`, `invoke/stream/batch`, Tool Calling 기초
- LangChain V0의 메시지 기본 개념


## 메시지란 무엇인가요?

메시지(Message)는 LangChain에서 LLM과 주고받는 대화의 기본 단위예요. 에이전트의 모든 상호작용은 메시지 리스트로 구성되고, LangGraph의 상태(State)에 저장돼요.

각 메시지는 다음 세 가지 핵심 요소로 구성돼요:

| 요소 | 역할 | 예시 |
|------|------|------|
| **역할(Role)** | 메시지 발신자 구분 | `system`, `user`, `assistant`, `tool` |
| **콘텐츠(Content)** | 실제 메시지 내용 | 텍스트, 이미지, 오디오 등 |
| **메타데이터(Metadata)** | 추가 정보 | `name`, `id`, 토큰 사용량, 도구 호출 정보 |

```mermaid
flowchart TD
    A["사용자 입력"]:::input --> B["HumanMessage"]:::input
    C["시스템 설정"]:::process --> D["SystemMessage"]:::process
    B --> E["LLM 모델"]:::process
    D --> E
    E --> F["AIMessage"]:::output
    F --> G{"도구 호출?"}:::process
    G -->|"Yes: tool_calls"| H["도구 실행"]:::storage
    H --> I["ToolMessage"]:::storage
    I --> E
    G -->|"No: content"| J["최종 응답"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### LangGraph에서 메시지의 위치

LangGraph 에이전트의 상태에는 항상 `messages` 필드가 있어요. 각 노드(Node)는 이 리스트에서 메시지를 읽고 새로운 메시지를 추가해요.

> 🔑 **핵심 개념**: LangGraph에서 `add_messages` reducer는 메시지 리스트를 자동으로 관리해요. 같은 `id`를 가진 메시지는 업데이트되고, 새로운 메시지는 추가돼요. 다음 노트북 `04-StateGraph-Basics.ipynb`에서 자세히 다룰 거예요.


## 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 로드해요
from dotenv import load_dotenv
import os

# .env 파일의 환경 변수를 로드해요
load_dotenv(override=True)

# LangSmith 추적 설정 (선택 사항)
# LangSmith에서 메시지 흐름을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part03-Messages"


## 1. 메시지 기본 사용법

메시지 객체를 생성하고 모델에 리스트로 전달하는 가장 기본적인 패턴이에요. 메시지 리스트를 통해 대화 컨텍스트를 유지할 수 있어요.

### 메시지 Import 경로 (V1)

LangChain V1에서는 모든 메시지를 `langchain.messages`에서 가져와요.

| V1 (올바른 방법) | 이전 방식 (피해야 할 방법) |
|-----------------|---------------------------|
| `from langchain.messages import HumanMessage` | `from langchain_core.messages import HumanMessage` |


In [ ]:
# ---------------------------------------------------
# V1 메시지 Import 및 모델 초기화
# ---------------------------------------------------
# V1 통합 경로: langchain.messages (langchain_core 대신)
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델: "anthropic:claude-sonnet-4-5", "google_genai:gemini-2.0-flash"
model = init_chat_model("openai:gpt-4o-mini")

print(f"모델 초기화 완료: {type(model).__name__}")


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 객체를 사용한 기본 모델 호출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 딕셔너리 형식으로 메시지 구성하기

메시지 객체 대신 딕셔너리로도 메시지를 지정할 수 있어요. `role` 키에는 역할을, `content` 키에는 내용을 넣어요. JSON 직렬화가 필요하거나 외부 API 연동 시 편리해요.

| 딕셔너리 `role` | 메시지 객체 |
|---|---|
| `system` | `SystemMessage` |
| `user` | `HumanMessage` |
| `assistant` | `AIMessage` |
| `tool` | `ToolMessage` |


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 딕셔너리 형식으로 메시지 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 4가지 메시지 타입 상세 가이드

LangChain V1은 4가지 핵심 메시지 타입을 제공해요. 각 타입은 대화에서 명확한 역할을 담당해요.

| 메시지 타입 | 역할 | 대화에서 위치 |
|------------|------|---------------|
| `SystemMessage` | 모델 동작 방식과 역할 정의 | 대화 시작 부분 |
| `HumanMessage` | 사용자 입력 (텍스트 + 멀티모달) | 사용자 차례마다 |
| `AIMessage` | 모델 생성 응답 (텍스트 + 도구 호출) | 모델 응답마다 |
| `ToolMessage` | 도구 실행 결과 전달 | 도구 호출 직후 |


### 2-1. SystemMessage — 모델 행동 지침

`SystemMessage`는 모델의 페르소나, 응답 스타일, 제약 조건 등을 정의해요. 효과적인 시스템 메시지는 모델이 일관된 방식으로 동작하도록 만들어요.

> 🔑 **핵심 개념**: 시스템 메시지는 대화 전체에 영향을 미치는 "설정" 이에요. 역할(`role`), 제약(`constraint`), 형식(`format`) 세 가지 요소를 포함하는 것이 좋아요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: SystemMessage: 모델 역할과 동작 방식 지정
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2-2. HumanMessage — 사용자 입력

`HumanMessage`는 사용자가 보내는 모든 입력을 담아요. 텍스트뿐 아니라 이미지, 오디오 등 멀티모달 콘텐츠도 포함할 수 있어요.


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: HumanMessage: 기본 생성 및 메타데이터 추가
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메타데이터가 포함된 메시지로 모델 호출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2-3. AIMessage — 모델 응답

`AIMessage`는 모델이 생성한 응답이에요. 텍스트 응답 외에도 도구 호출(`tool_calls`) 정보, 토큰 사용량(`usage_metadata`), 제공자 메타데이터를 포함해요.


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: AIMessage 구조 상세 분석
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 다중 턴 대화: AIMessage를 이전 대화에 포함시키기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. ToolMessage — 도구 호출 흐름

`ToolMessage`는 도구 실행 결과를 모델에 다시 전달하는 특별한 메시지예요. LangGraph 에이전트의 ReAct 루프에서 핵심 역할을 해요.

도구 호출 전체 흐름은 아래와 같이 4단계로 이루어져요:

```mermaid
sequenceDiagram
    participant U as 사용자
    participant M as LLM 모델
    participant T as 도구 실행

    U->>M: HumanMessage (서울 날씨 알려줘)
    M-->>T: AIMessage (tool_calls: get_weather, id=call_123)
    T-->>M: ToolMessage (content: 맑음 10도, tool_call_id=call_123)
    M-->>U: AIMessage (content: 서울은 현재 맑고 10도입니다)
```

### AIMessage의 두 가지 모드

| 모드 | `content` | `tool_calls` | 의미 |
|------|-----------|-------------|------|
| 텍스트 응답 | `"서울의 수도는..."` | `[]` (빈 리스트) | LLM이 직접 답변 |
| 도구 호출 | `""` (빈 문자열) | `[{name, args, id}]` | LLM이 도구 사용을 요청 |


In [ ]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ToolMessage 수동 구성: tool_call_id 매칭 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2단계: 도구 실행 결과를 ToolMessage로 래핑
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3단계: 전체 도구 호출 대화 흐름 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 실제 도구 바인딩과 자동 도구 호출 확인

수동으로 메시지를 구성하는 방법을 이해했어요. 이제 실제로 모델에 도구를 바인딩하고, 모델이 자동으로 도구를 선택하는 과정을 확인해볼게요.


In [ ]:
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 실제 도구 바인딩 후 AIMessage 구조 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: tool_call_id 매칭으로 완전한 ReAct 루프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 메시지 콘텐츠 형식 — 텍스트부터 멀티모달까지

메시지의 `content`는 단순 문자열부터 복잡한 멀티모달 블록까지 다양한 형식을 지원해요. LangChain V1에서는 두 가지 멀티모달 형식을 제공해요.

| 형식 | 특징 | 사용 시점 |
|------|------|----------|
| **문자열** | 단순 텍스트 | 기본 대화, 텍스트 처리 |
| **Provider 네이티브** | `image_url` 형식 | OpenAI, Anthropic 특정 기능 사용 시 |
| **표준 content_blocks** | `url` 형식 (V1 신기능) | 모든 제공자에서 동일하게 동작 |


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 1: 문자열 (가장 기본)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import urllib.request
import base64

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 2: Provider 네이티브 (image_url 형식)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 콘텐츠 형식 3: 표준 content_blocks (V1 신기능 - 권장)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 멀티모달 메시지로 모델 호출 (gpt-4o-mini는 비전 지원)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 표준 content_blocks로 동일한 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 메시지 유틸리티 함수

LangGraph에서 자주 사용하는 메시지 처리 패턴들을 배워봐요. `pretty_print()`는 디버깅 시 메시지 구조를 빠르게 확인할 때 유용해요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: pretty_print(): 메시지를 보기 좋게 출력해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 정보 출력 헬퍼 함수
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 실습: 멀티 에이전트 대화 구성

지금까지 배운 개념을 종합해서 직접 대화를 구성해봐요. `name` 필드를 사용하여 두 에이전트가 주고받는 대화를 만들어보세요.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# ============================================================
# TODO: 아래 코드를 완성해서 두 에이전트 대화를 구성해보세요
# 힌트: name 필드를 사용해서 "researcher"와 "analyst" 에이전트를 구분해요
#       각 에이전트는 HumanMessage 혹은 AIMessage에 name을 지정해요
# 예상 결과: 두 에이전트 간의 자연스러운 대화 흐름이 출력되어야 해요
# ============================================================


# TODO: 아래 대화를 완성해보세요
# researcher 에이전트가 분석 결과를 전달하고
# analyst 에이전트가 해석을 제공하는 시나리오예요


    # TODO: researcher 에이전트의 메시지를 추가해보세요
    # 힌트: HumanMessage(content="...", name="researcher")
    # HumanMessage(...),

    # TODO: analyst 에이전트의 응답을 추가해보세요
    # 힌트: AIMessage(content="...", name="analyst")
    # AIMessage(...),

    # TODO: 새로운 질문을 추가해보세요
    # HumanMessage(...),


# TODO: 대화를 모델에 전달하고 결과를 출력해보세요
# response = model.invoke(multi_agent_conversation)
# print(response.content)

# TODO: 위의 주석을 해제하고 코드를 완성해보세요!
# name 필드로 에이전트를 구분하는 패턴은 Multi-Agent 파트에서 자주 사용돼요.


## 7. 메시지 타입 확인과 대화 이력 관리

실무에서는 메시지 타입을 확인하거나 필터링해야 할 경우가 자주 생겨요. 특히 LangGraph 상태(State)에서 메시지를 처리할 때 유용한 패턴들을 알아봐요.


In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 타입 확인 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 대화 이력 요약 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **4가지 메시지 타입**: `SystemMessage`(모델 지침), `HumanMessage`(사용자 입력), `AIMessage`(모델 응답), `ToolMessage`(도구 결과) 각각의 역할과 사용 시점
- **메시지 메타데이터**: `name`으로 발신자를 구별하고, `id`로 메시지를 고유 추적해요. LangGraph의 `add_messages` reducer와 연계돼요
- **딕셔너리 형식**: `{"role": "user", "content": "..."}` 형식으로 간결하게 메시지를 구성해요. 역할 이름은 `system`, `user`, `assistant`, `tool`을 사용해요
- **ToolMessage ID 매칭**: `tool_call_id`는 반드시 `AIMessage`의 `tool_calls[i]['id']`와 일치해야 해요. ID 불일치는 가장 흔한 도구 호출 오류예요
- **멀티모달 콘텐츠**: Provider 네이티브(`image_url`) 방식과 표준 `content_blocks`(`url`) 방식 두 가지를 지원해요. 새 코드에서는 표준 방식을 권장해요
- **V1 import 경로**: `from langchain.messages import ...` (langchain_core 대신 langchain 사용)


## 다음 노트북 예고

다음 `04-StateGraph-Basics.ipynb`에서는 **StateGraph, State, Node, Edge, START/END**를 배워요. 이번 노트북에서 배운 메시지 타입들이 LangGraph의 상태(State)에 어떻게 저장되고 관리되는지, `add_messages` reducer가 메시지 ID를 어떻게 활용하는지 직접 확인할 수 있어요.
